# 跟踪基础

### 设置

请确保设置您的环境变量，包括您的 OpenAI API 密钥。

In [ ]:
# 您可以在代码中直接设置
import os
os.environ["OPENAI_API_KEY"] = ""
os.environ["LANGSMITH_API_KEY"] = ""
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "langsmith-academy"

In [ ]:
# 或者您可以使用 .env 文件
from dotenv import load_dotenv
load_dotenv(dotenv_path="../../.env", override=True)

### 使用 @traceable 进行跟踪

@traceable 装饰器是从 LangSmith Python SDK 记录跟踪的简单方法。只需用 @traceable 装饰任何函数即可。

该装饰器的工作原理是每次调用函数时为您创建一个运行树，并将其插入到当前跟踪中。然后将函数输入、名称和其他信息流式传输到 LangSmith。如果函数引发错误或返回响应，该信息也会添加到树中，并将更新补丁到 LangSmith，以便您可以检测和诊断错误源。这一切都在后台线程中完成，以避免阻塞您的应用程序执行。

In [ ]:
# TODO: 导入 traceable
from openai import OpenAI
from typing import List
import nest_asyncio
from utils import get_vector_db_retriever

MODEL_PROVIDER = "openai"
MODEL_NAME = "gpt-4o-mini"
APP_VERSION = 1.0
RAG_SYSTEM_PROMPT = """您是一个问答任务的助手。
使用以下检索到的上下文片段来回答对话中的最新问题。
如果您不知道答案，请直接说您不知道。
最多使用三句话，保持答案简洁。
"""

openai_client = OpenAI()
nest_asyncio.apply()
retriever = get_vector_db_retriever()

# TODO: 为每个函数设置跟踪
def retrieve_documents(question: str):
    return retriever.invoke(question)   # 注意：这是一个 LangChain 向量数据库检索器，所以这个 .invoke() 调用将自动被跟踪


def generate_response(question: str, documents):
    formatted_docs = "\n\n".join(doc.page_content for doc in documents)
    messages = [
        {
            "role": "system",
            "content": RAG_SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": f"上下文: {formatted_docs} \n\n 问题: {question}"
        }
    ]
    return call_openai(messages)


def call_openai(
    messages: List[dict], model: str = MODEL_NAME, temperature: float = 0.0
) -> str:
    return openai_client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
    )


def langsmith_rag(question: str):
    documents = retrieve_documents(question)
    response = generate_response(question, documents)
    return response.choices[0].message.content

@traceable 为您处理 RunTree 生命周期！

In [ ]:
question = "如何使用 @traceable 装饰器进行跟踪？"
ai_answer = langsmith_rag(question)
print(ai_answer)

##### 让我们在 LangSmith 中查看一下！

### 添加元数据

LangSmith 支持在跟踪中发送任意元数据。

元数据是可以附加到运行的键值对集合。元数据可用于存储有关运行的附加信息，例如生成运行的应用程序版本、生成运行的环境或您想要与运行关联的任何其他信息。与标签类似，您可以使用元数据在 LangSmith UI 中过滤运行，并可用于将运行分组在一起进行分析。

In [ ]:
from langsmith import traceable

@traceable(
    # TODO: 添加元数据
    # metadata={"vectordb": "sklearn"}
)
def retrieve_documents(question: str):
    return retriever.invoke(question)

@traceable
def generate_response(question: str, documents):
    formatted_docs = "\n\n".join(doc.page_content for doc in documents)
    messages = [
        {
            "role": "system",
            "content": RAG_SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": f"上下文: {formatted_docs} \n\n 问题: {question}"
        }
    ]
    return call_openai(messages)

@traceable(
    # TODO: 添加元数据
    # metadata={"model_name": MODEL_NAME, "model_provider": MODEL_PROVIDER}
)
def call_openai(
    messages: List[dict], model: str = MODEL_NAME, temperature: float = 0.0
) -> str:
    return openai_client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
    )

@traceable
def langsmith_rag(question: str):
    documents = retrieve_documents(question)
    response = generate_response(question, documents)
    return response.choices[0].message.content

In [ ]:
question = "如何使用 @traceable 向运行添加元数据？"
ai_answer = langsmith_rag(question)
print(ai_answer)

您还可以在运行时添加元数据！

In [ ]:
question = "如何在运行时添加元数据？"
ai_answer = langsmith_rag(question, langsmith_extra={"metadata": {"runtime_metadata": "foo"}})
print(ai_answer)

##### 让我们在 LangSmith 中查看一下！